In [2]:
import pandas as pd
import numpy as np

In [9]:
datas = ["LINCS", "CDRP-bio"]
id = 1
IN_CSV  = f"summary_{datas[id]}_torch_mlp_topn.csv"
OUT_CSV = f"summary_{datas[id]}.csv"

MOD_ORDER  = ["CP", "GE", "Early Fusion", "Late Fusion"]
TOPN_ORDER = [5,10,15,20,25,30,50,75,100,150,200,300,400,978]  # 필요하면 수정

df = pd.read_csv(IN_CSV)

# 필수 컬럼 체크
req = {"dataset", "top_n", "fold", "modality", "f1"}
missing = req - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in {IN_CSV}: {missing}")

# top_n을 int로 (가능하면)
def _to_int(x):
    try:
        return int(x)
    except Exception:
        return x

df["top_n"] = df["top_n"].apply(_to_int)

# (원하면 특정 dataset만)
# df = df[df["dataset"] == "LINCS"].copy()

# fold별 F1 mean/std
g = (
    df.groupby(["dataset", "top_n", "modality"], as_index=False)
      .agg(
          f1_mean=("f1", "mean"),
          f1_std =("f1", "std"),
      )
)

# wide로 펼치기
wide = g.pivot(index=["dataset", "top_n"], columns="modality")
# (metric, modality) -> "CP_F1 mean" 같은 단일 헤더로
wide.columns = [f"{mod}_F1 {met.split('_')[1]}" for met, mod in wide.columns]  # mean/std
wide = wide.reset_index()

# 원하는 컬럼 순서 강제
target_cols = ["dataset", "top_n"]
for mod in MOD_ORDER:
    target_cols += [f"{mod}_F1 mean", f"{mod}_F1 std"]

for c in target_cols:
    if c not in wide.columns:
        wide[c] = np.nan

wide = wide[target_cols]

# top_n 정렬
order_map = {v: i for i, v in enumerate(TOPN_ORDER)}
wide["_rank"] = wide["top_n"].map(order_map).fillna(10**9).astype(int)
wide = wide.sort_values(["dataset", "_rank", "top_n"]).drop(columns=["_rank"])

wide.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
print("Shape:", wide.shape)


Saved: summary_CDRP-bio.csv
Shape: (10, 10)
